In [45]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

from tensorgrad import Tensor, Layer, MLP, unbroadcast, gradcheck

In [52]:
BATCH_SIZE = 100

train_data = datasets.MNIST(root='../data',
                            download=True,
                            train=True,
                            transform=transforms.Lambda(lambda x : np.array(x).flatten()))

test_data = datasets.MNIST(root='../data',
                           download=True,
                           train=False,
                           transform=transforms.Lambda(lambda x : np.array(x).flatten()))

x_train, y_train = [], []
x_batch, y_batch = [], []
x_val, y_val = [], []

for i, (x, y) in enumerate(train_data, 1):
    if i > 0 and i % BATCH_SIZE == 0:
        if i <= 50000:
            x_train.append(Tensor(np.array(x_batch)))
            y_train.append(Tensor(np.array(y_batch)))
        else:
            x_val.append(Tensor(np.array(x_batch)))
            y_val.append(Tensor(np.array(y_batch)))
        x_batch, y_batch = [], []

    x_batch.append(x / 255)
    y_batch.append(np.eye(10)[y])

x_test, y_test = [], []
x_batch, y_batch = [], []

for i, (x, y) in enumerate(test_data, 1):
    if i > 0 and i % BATCH_SIZE == 0:
        x_test.append(Tensor(np.array(x_batch)))
        y_test.append(Tensor(np.array(y_batch)))
        x_batch, y_batch = [], []
    x_batch.append(x / 255)
    y_batch.append(np.eye(10)[y])

len(x_train), len(y_train), len(x_val), len(y_val), len(x_test), len(y_test)

(500, 500, 100, 100, 100, 100)

In [53]:
EPOCHS = 100
LEARNING_RATE = 1e-2

model = MLP()

for epoch in range(1, EPOCHS + 1):
    for x, y in zip(x_train, y_train):
        y_pred = model(x)
        loss = y_pred.cross_entropy(y.data)
        model.zero_grad()
        loss.backward()
        model.step(LEARNING_RATE)

    if epoch % 10 == 0:
        print(f'epoch: {epoch}, loss: {loss.data:.4f}')

epoch: 10, loss: 0.2255
epoch: 20, loss: 0.1305
epoch: 30, loss: 0.0830
epoch: 40, loss: 0.0590
epoch: 50, loss: 0.0464
epoch: 60, loss: 0.0396
epoch: 70, loss: 0.0349
epoch: 80, loss: 0.0298
epoch: 90, loss: 0.0251
epoch: 100, loss: 0.0202


In [54]:
total = len(test_data)
correct = 0
for x, y in zip(x_test, y_test):
    y_pred = model(x)
    correct += np.sum(np.argmax(y.data, -1) == np.argmax(y_pred.data, -1))

print(f'loss: {loss.data:.4f}, accuracy: {correct}/{total} = {correct/total*100:.4f}%')

loss: 0.0202, accuracy: 9775/10000 = 97.7500%
